# 01 – Аудит данных Северной Америки

Цель – установить состав и ограничения исходной выборки перед подготовкой собственной базы данных

Одна запись описывает посадку, которая может включать несколько попыток захода. `has_ga` означает наличие хотя бы одного ухода на второй круг, а не аварийность или нарушение

Источник: [Large Landing Trajectory Data Set for Go-Around Analysis](https://zenodo.org/records/7148117), DOI 10.5281/zenodo.7148117, версия 1.0, 2022. Наблюдения относятся к 2019 году и получены авторами из OpenSky. Погода добавлена из Iowa Environmental Mesonet

После регионального разделения выполнялась первичная диагностика. Этот ноутбук повторно проверяет исходный CSV и показывает результаты без изменения записей. Визуализация и разбор графиков запланированы следующим этапом


In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display

# Находим проект независимо от папки запуска Jupyter
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "north_america" / "data" / "source" / "north_america.csv").exists())
REGION = ROOT / "north_america"
SOURCE = REGION / "data" / "source" / "north_america.csv"
REPORTS = REGION / "reports"
summary = json.loads((REPORTS / "profile_summary.json").read_text(encoding="utf-8"))

def sha256(path):
    result = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024**2), b""):
            result.update(block)
    return result.hexdigest()

hash_before = sha256(SOURCE)
assert hash_before == summary["source_sha256"], "CSV отличается от ранее проверенного файла"
print("SHA256:", hash_before)


SHA256: d9443153e2ead53b857ece9a1bcb126e461ea6b5bc90a69f89eb7092bdb34c19


## Состав и период наблюдений

Читаем CSV частями как строки, чтобы не менять исходные значения. Пропуском считаем пустое поле, а строковый список `[]` сохраняет свое значение


In [2]:
rows = 0
go_arounds = 0
airport_codes = set()
country_codes = set()
missing_counts = None
key_parts = []
earliest = latest = None
time_errors = 0
unexpected_target = 0
columns = None

# Полная проверка файла без загрузки всех 31 полей в память одновременно
for chunk in pd.read_csv(SOURCE, dtype=str, keep_default_na=False, chunksize=100_000):
    columns = chunk.columns.tolist()
    rows += len(chunk)
    go_arounds += int(chunk["has_ga"].eq("True").sum())
    unexpected_target += int((~chunk["has_ga"].isin(["True", "False"])).sum())
    airport_codes.update(chunk["airport"])
    country_codes.update(chunk["airport_country"])
    current_missing = chunk.eq("").sum()
    missing_counts = current_missing if missing_counts is None else missing_counts.add(current_missing)
    times = pd.to_datetime(chunk["time"], errors="coerce", utc=True)
    time_errors += int(times.isna().sum())
    if times.notna().any():
        earliest = times.min() if earliest is None else min(earliest, times.min())
        latest = times.max() if latest is None else max(latest, times.max())
    key_parts.append(chunk[["time", "icao24", "airport"]])

keys = pd.concat(key_parts, ignore_index=True)
duplicate_keys = int(keys.duplicated().sum())
del key_parts, keys
assert rows == summary["rows"]
assert go_arounds == summary["go_arounds"]
assert len(airport_codes) == summary["airports"]
assert len(country_codes) == summary["countries"]
display(pd.DataFrame({"Показатель": ["Записей", "Полей", "Аэропортов", "Стран и территорий", "Записей с уходом", "Доля записей с уходом, %", "Начало периода UTC", "Конец периода UTC", "Ошибки времени", "Неожиданные значения has_ga", "Повторы ключа time + icao24 + airport"],
                      "Значение": [rows, len(columns), len(airport_codes), len(country_codes), go_arounds, round(100 * go_arounds / rows, 4), str(earliest), str(latest), time_errors, unexpected_target, duplicate_keys]}))


,Показатель,Значение
0,Записей,3909316
1,Полей,31
2,Аэропортов,74
3,Стран и территорий,6
4,Записей с уходом,16288
5,"Доля записей с уходом, %",0.4166
6,Начало периода UTC,2019-01-01 00:01:13+00:00
7,Конец периода UTC,2019-12-30 23:59:59+00:00
8,Ошибки времени,0
9,Неожиданные значения has_ga,0


Совпадение выбранного ключа проверяет только точные совпадения времени, борта и аэропорта. При подключении другого источника нужны также сопоставление временных интервалов и разбор конфликтов. Отсутствие повторов ключа не доказывает правильность разметки

In [3]:
countries = pd.read_csv(REPORTS / "country_summary.csv").sort_values("rows", ascending=False)
assert int(countries["rows"].sum()) == rows
assert int(countries["go_arounds"].sum()) == go_arounds
countries["sample_percent"] = 100 * countries["rows"] / rows
display(countries.round(4))


,airport_country,rows,go_arounds,ga_percent,sample_percent
5,USA,3850534,16010,0.4158,98.4964
3,CAN,50595,245,0.4842,1.2942
4,MEX,8047,33,0.4101,0.2058
0,ABW,107,0,0.0000,0.0027
1,BES,32,0,0.0000,0.0008
2,BHS,1,0,0.0000,0.0000


США составляют около 98,5% выборки. Канада и Мексика представлены значительно слабее, Карибские территории – единичными наблюдениями. Выборка не является равномерным представлением Северной Америки. Географическая группировка не задает общих авиационных правил для всех стран

## Поля, пропуски и числовые диапазоны

[Русский словарь всех 31 полей](../../docs/data_dictionary.md) описывает смысл, заявленные единицы и ограничения. Проценты в общем словаре относятся ко всему исходному набору; ниже показана именно американская выборка


In [4]:
missing = pd.DataFrame({"missing": missing_counts, "missing_percent": 100 * missing_counts / rows})
previous_missing = pd.read_csv(REPORTS / "missingness.csv", index_col=0)
assert missing["missing"].astype(int).equals(previous_missing["missing"].astype(int))
display(missing.sort_values("missing_percent", ascending=False).round(4))
display(pd.read_csv(REPORTS / "numeric_ranges.csv", index_col=0))


,missing,missing_percent
wind_gust_knts,3333863,85.2800
icaoaircrafttype,473231,12.1052
wtc,362357,9.2691
typecode,360047,9.2100
glide_slope_angle,321840,8.2326
rwy_length,321840,8.2326
has_intersection,321840,8.2326
press_sea_level_p,237010,6.0627
operator_country,218607,5.5920
operator_region,218607,5.5920


,count,missing,parse_errors,min,max
n_approaches,3909316,0,0,1.000000,22.000000
n_rwy_approached,3909316,0,0,1.000000,5.000000
glide_slope_angle,3587476,321840,0,2.750000,3.540000
rwy_length,3587476,321840,0,1064.720999,4875.435865
wind_speed_knts,3903034,6282,0,0.000000,96.000000
wind_dir_deg,3761475,147841,0,0.000000,360.000000
wind_gust_knts,575453,3333863,0,12.000000,60.000000
visibility_m,3908647,669,0,0.000000,96560.640000
temperature_deg,3908908,408,0,-33.300000,2018.000000
press_sea_level_p,3672306,237010,0,969.800000,1048.400000


Порывы ветра отсутствуют примерно в 85,28% записей. Отсутствие сообщения о порывах нельзя автоматически считать нулем

`press_p` заявлен авторами как QNH в гПа. `press_sea_level_p` – отдельное поле давления на уровне моря. QFE в схеме отсутствует. По этим полям нельзя установить фактическую настройку высотомера экипажа

Единица `rwy_length` требует проверки: авторы указывают километры, но значения 1064,72–4875,44 похожи на метры. Пока единицы не подтверждены, конвертацию не выполняем

Максимальная видимость 96560,64 м требует проверки исходной сводки и способа преобразования. Нахождение внутри диагностического диапазона не подтверждает корректность значения


## Диагностические флаги и открытые вопросы

Первичная проверка использовала ориентировочные границы: температура от −80 до 60 °C, давление 850–1100 гПа, ветер 0–150 узлов, направление 0–360°, видимость 0–100000 м. Границы используются для поиска аномалий в табличных данных. Исходные сводки METAR в CSV отсутствуют


In [5]:
flags = pd.read_csv(REPORTS / "review_flag_summary.csv", index_col=0)
display(flags.loc[flags["rows"] > 0])
flagged = pd.read_csv(REGION / "data" / "review" / "flagged_rows.csv")
assert len(flagged) == summary["unique_flagged_rows"]
suspect = flagged.loc[pd.to_numeric(flagged["temperature_deg"], errors="coerce").eq(2018.0)].copy()
suspect_times = pd.to_datetime(suspect["time"], utc=True)
display(pd.DataFrame({"Показатель": ["Температура 2018,0: записей", "Аэропортов", "Первое время UTC", "Последнее время UTC"],
                      "Значение": [len(suspect), suspect["airport"].nunique(), str(suspect_times.min()), str(suspect_times.max())]}))
display(suspect[["source_row", "time", "airport", "temperature_deg", "has_ga"]].head(10))


,rows,go_arounds,percent
temperature_deg:outside_diagnostic_range,260,0,0.006651


,Показатель,Значение
0,"Температура 2018,0: записей",260
1,Аэропортов,33
2,Первое время UTC,2019-07-20 17:06:21+00:00
3,Последнее время UTC,2019-07-20 17:44:59+00:00


,source_row,time,airport,temperature_deg,has_ga
0,22987,2019-07-20 17:09:59+00:00,KAUS,2018.0,False
1,22988,2019-07-20 17:25:59+00:00,KAUS,2018.0,False
2,83849,2019-07-20 17:16:46+00:00,KBOS,2018.0,False
3,83850,2019-07-20 17:18:14+00:00,KBOS,2018.0,False
4,83851,2019-07-20 17:19:21+00:00,KBOS,2018.0,False
5,83852,2019-07-20 17:20:53+00:00,KBOS,2018.0,False
6,83853,2019-07-20 17:27:32+00:00,KBOS,2018.0,False
7,83854,2019-07-20 17:28:59+00:00,KBOS,2018.0,False
8,83855,2019-07-20 17:30:38+00:00,KBOS,2018.0,False
9,83856,2019-07-20 17:32:22+00:00,KBOS,2018.0,False


260 записей со значением температуры 2018,0 встречаются в 33 аэропортах США 20 июля 2019 года в интервале 17:06:21–17:44:59 UTC. Возможна общая проблема преобразования метеоданных, но ее причина не установлена. Значение не заменяем догадкой

По диагностическим границам давления флагов нет. Это не подтверждение корректности QNH во всех строках. Для проверки потребуется исходный METAR, станция, время, единицы и правило преобразования

Нужно отдельно проверить категориальные погодные коды, полноту покрытия, соответствие ВПП и качество меток `has_ga`. Эти проверки еще не завершены


## Официальные документы и научные источники

Собрана начальная подборка для интерпретации данных и проверки аномалий. Найденный документ не равен изученному положению. При использовании нормы фиксируем конкретный пункт, юрисдикцию, редакцию и дату применимости

| Источник | Текущий статус |
| --- | --- |
| FAA Aviation Weather Handbook 28B, 2026 | PDF скачан, титульные сведения проверены, тематические страницы найдены |
| FAA AIM Basic 2025 + Changes 1–3 | PDF скачан, актуальная подборка с 09.07.2026 |
| Transport Canada AIM 2026-2 | PDF скачан |
| AIP Dutch Caribbean AIRAC AMDT 03-26 | PDF скачан, применимость к конкретному аэропорту проверяем отдельно |
| FAA JO 7110.65BB | Официальный онлайн-источник найден, нужные пункты еще не разобраны |
| ICAO Annex 3 и PANS-MET Doc 10157 | Найдены сведения о редакциях, полные тексты не изучены |
| ICAO Annex 11 | Найден официальный указатель на бесплатную онлайн-версию, редакцию предстоит проверить |
| ICAO Doc 4444 | Применимый полный текст пока не проверен |
| Мексика | Портал AIP найден, полный доступ не получен |
| Багамы | Официальные ссылки CAR MET, ATS, CNS найдены, редакции не проверены |
| Исторические нормы и схемы 2019 года | Подборка не завершена |

Реестры: [официальные документы региона](../documents/README.md), [общие источники](../../docs/official_sources.md), [скачанные PDF](../documents/download_manifest.json)

Найдены публикации о прогнозировании уходов, анализе ADS-B, нестабилизированных заходах и рисках посадки:

| Работа | DOI | Степень изучения |
| --- | --- | --- |
| Dai, Liu, Hansen, 2021, Modeling go-around occurrence | [10.1016/j.trc.2021.103262](https://doi.org/10.1016/j.trc.2021.103262) | Сведения и аннотация |
| Kumar и др., 2021, Classification and Analysis of Go-Arounds | [10.3390/aerospace8100291](https://doi.org/10.3390/aerospace8100291) | PDF скачан, отдельные разделы |
| Dai, Liu, Hansen, 2022, Sequential Prediction | [10.2139/ssrn.4019517](https://doi.org/10.2139/ssrn.4019517) | Препринт, сведения и аннотация |
| Misra и др., 2022, Assessing Unstabilized Approaches | [10.7771/2159-6670.1234](https://doi.org/10.7771/2159-6670.1234) | Аннотация |
| Odisho, Truong, Joslin, 2022, Runway Excursion Risk Mitigation | [10.2514/1.I010972](https://doi.org/10.2514/1.I010972) | Сведения и описание |
| Puranik, Rodriguez, Mavris, 2020, Safety-critical landing metrics | [10.1016/j.trc.2020.102819](https://doi.org/10.1016/j.trc.2020.102819) | Библиографические сведения |

Также найдена [диссертация Lu Dai, Berkeley, 2022](https://escholarship.org/uc/item/87453364). Локальное скачивание получило HTTP 403. Карточки книг Flight Theory and Aerodynamics, 4-е издание, 2021, и Human Factors in Aviation and Aerospace, 3-е издание, 2022, найдены, полные тексты не изучены

Научные статьи помогают выбирать методы и формулировать гипотезы, но не заменяют обязательные нормы. Современные документы не применяем автоматически к наблюдениям 2019 года


## Ограничения и порядок решений

В CSV нет траекторий, высоты самолета, IAS, разрешений УВД, текста ATIS и идентификатора процедуры захода. Проверить соблюдение переходной высоты, скоростных ограничений и RNAV/RNP по этой таблице нельзя. Скорость ветра не является скоростью самолета, а справочный угол ILS не является измеренным углом его движения

Для каждой группы аномалий фиксируем записи, гипотезу, проверку первоисточника и решение автора проекта. Возможные решения: оставить, исправить по подтвержденному источнику, добавить флаг, исключить из конкретного анализа или отложить решение. Исходные записи сохраняем во всех случаях

При добавлении нового набора сначала проверяем пересечения полетов. Данные о том же полете могут обогащать существующую запись, а не создавать новый полет. NOAA и исходные METAR рассматриваются как будущие источники проверки погоды; пока они не подключены


In [6]:
hash_after = sha256(SOURCE)
assert hash_after == hash_before, "Исходный CSV изменился во время аудита"
print("Исходный CSV сохранен без изменений")
print("Ноутбук не удаляет строки и не исправляет значения")


Исходный CSV сохранен без изменений
Ноутбук не удаляет строки и не исправляет значения


## Вывод аудита

Американская выборка содержит 3 909 316 записей, 31 поле, 74 аэропорта и 6 стран и территорий. В 16 288 записях отмечен хотя бы один уход на второй круг. Покрытие преимущественно относится к США

Обнаружена группа подозрительных температур, остаются вопросы к единицам длины ВПП, кодированию видимости, интерпретации давления и пропусков. Первичная диагностика не является завершенной очисткой или независимой проверкой разметки

Следующий этап – последовательно разобрать графики и группы аномалий, проверить исходные наблюдения и согласовать решения. Подготовленная версия собственной базы формируется только после этого


# Обоснование состава аналитической выборки

## Критерий точности

Единица учета – запись о посадке, событие – наличие хотя бы одного отмеченного ухода

$$
\widehat p_i=\frac{k_i}{n_i},\qquad
\Delta_i=\max(\widehat p_i-L_i,\;U_i-\widehat p_i),\qquad
\Delta_i\leq\varepsilon
$$

$[L_i,U_i]$ – двусторонний интервал Уилсона с номинальным уровнем доверия 95%, $z=\Phi^{-1}(0{,}975)$. $\varepsilon$ – допустимое абсолютное отклонение в единицах вероятности. В таблицах $\Delta$ выражено в процентных пунктах

Согласован допуск $\varepsilon=0{,}002$ (0,2 п.п.), дата 07.10.2026. Интервалы рассчитаны для всех 74 аэропортов и шести стран и территорий, включая исключенные. Это выбранная абсолютная точность исследования. Параметр согласован после предварительного анализа, поэтому применение является ретроспективным

Расчет предполагает независимость наблюдений и общую вероятность события внутри группы. Полнота периода, зависимость полетов и корректность разметки оцениваются отдельно. Низкая точность отдельной оценки не требует удаления записей из исходной базы

## Исключенные страны и территории

| Группа | Записей | С уходом | Интервал Уилсона, % | $\Delta$, п.п. | Основание текущего решения |
| --- | ---: | ---: | --- | ---: | --- |
| Аруба (ABW) | 107 | 0 | 0.0000–3.4657 | 3.4657 | Критерий точности не выполнен; записи сохранены |
| Карибы (BES) | 32 | 0 | 0.0000–10.7179 | 10.7179 | Критерий точности не выполнен; записи сохранены |
| Багамы (BHS) | 1 | 0 | 0.0000–79.3451 | 79.3451 | Критерий точности не выполнен; одна запись |
| Мексика (MEX) | 8047 | 33 | 0.2922–0.5753 | 0.1652 | Национальный нормативный контекст не проверен; отдельное исследование отложено |

Малая доля страны в общей выборке не является самостоятельным доказательством нерепрезентативности. Исключение Мексики не объявляется следствием критерия точности. Национальные правила США и Канады на ее записи не переносятся

## Исключенные аэропорты США и Канады

| ICAO – город | Записей | С уходом | Интервал Уилсона, % | $\Delta$, п.п. |
| --- | ---: | ---: | --- | ---: |
| CYQB – Квебек | 1818 | 25 | 0.9332–2.0222 | 0.6470 |
| CYWG – Виннипег | 6 | 0 | 0.0000–39.0334 | 39.0334 |
| CYYJ – Виктория | 3112 | 32 | 0.7293–1.4480 | 0.4197 |
| KDEN – Денвер | 326 | 0 | 0.0000–1.1646 | 1.1646 |
| KJAX – Джэксонвилл | 7 | 0 | 0.0000–35.4330 | 35.4330 |
| KPIT – Питтсбург | 39 | 1 | 0.4541–13.1810 | 10.6169 |
| KPWM – Портленд, Мэн | 156 | 0 | 0.0000–2.4033 | 2.4033 |
| KRIC – Ричмонд | 1 | 0 | 0.0000–79.3451 | 79.3451 |
| KSYR – Сиракьюс | 10 | 0 | 0.0000–27.7533 | 27.7533 |

Шесть аэропортов США исключены по согласованному ограничению состава эксперимента группами с малым объемом наблюдений. Отсутствие уходов не использовано как самостоятельный критерий. Все шесть групп не проходят согласованный допуск. Формальный расчет дополняет первоначальное решение, принятое до согласования параметра

В Канаде текущий эксперимент ограничен CYUL – Монреаль и CYOW – Оттава, которые дают около 90,2% канадских наблюдений. CYYJ – Виктория и CYQB – Квебек имеют зарегистрированные события; меньший объем не доказывает их непригодность. CYWG – Виннипег представлен шестью записями. Все три группы не проходят согласованный допуск. Ограничение частотных оценок количественно подтверждено; исходные записи сохраняются

Критерий рассчитан для всех аэропортов, включая оставленные. Из 57 текущих аэропортов 11 не проходят допуск; дополнительные исключения не применены. Порог не подбирается под прежний список исключений. Ограничение точности аэропортной оценки и отбор записей для общей модели рассматриваются отдельно

## Сохранность и текущий состав

Исходный региональный CSV сохранен полностью. В аналитическом CSV 3 895 654 записи, 16 197 с отмеченным уходом, 55 аэропортов США и два аэропорта Канады. За пределами текущего анализа сохранены 13 662 записи

Значения температуры и других полей не удалены и не исправлены. Высокие доли уходов в KSFB – Санфорд и KCMA – Камарильо не стали основанием исключения; группы оставлены для проверки

Расчеты: `airport_precision.csv`, `country_precision.csv`. Повторение: `python north_america/scripts/assess_group_precision.py`

## Оставленные группы с ограниченной точностью

| Аэропорт | Записей | С уходом | $\Delta$, п.п. |
| --- | ---: | ---: | ---: |
| KCMA | 877 | 113 | 2.3803 |
| KMEM | 3426 | 25 | 0.3453 |
| KMKE | 3757 | 11 | 0.2308 |
| KMSY | 1535 | 6 | 0.4593 |
| KONT | 6458 | 31 | 0.2005 |
| KPBI | 2712 | 6 | 0.2606 |
| KPVD | 2773 | 16 | 0.3583 |
| KRSW | 890 | 5 | 0.7465 |
| KSAV | 995 | 18 | 1.0324 |
| KSFB | 1519 | 262 | 1.9819 |
| KTUL | 11291 | 164 | 0.2378 |

Эти 11 групп остаются в аналитическом CSV. Отдельные частотные оценки не проходят допуск 0,2 п.п. Решение о дальнейшем использовании требуется отдельно
